### Step 1: Blackbody 0D EBM

26/06/2026
We start with the most basic zero dimensional energy balance model. We consider our planet a perfect black body at this stage.

Obsv:
T_eq = 278.33 K
T_eq is exactly as expected

-----------------------------------------------------------------------------------
27/06/2026
We add an albedo to the planet. Since we are not setting up an albedo feedback yet, changing the parameter would be enough.

Obsv:
Temperature drops to 254.58K just as expected.

In [5]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

# Constants
sigma = 5.670374419e-8  # Stefan-Boltzmann constant (W m^-2 K-4)
PI = np.pi  # Pi

# Stellar Parameters

L_star = 3.828e26 # Solar Luminosity (W)

#Planetary Parameters

Orbital_radius = 1.496e11 # Orbital radius of Earth (m)
alpha = 0.3 # Albedo of the planet
epsilon0 = 0.77 # Emissivity of the planet

C = 4.2e8 # Heat capacity of the planet (J m^-2 K^-1)

The stellar constant ($S_0$) will not be taken as a constant but will be a function of Stellar Luminance ($L_*$) and distance from the star (r)

In [6]:
def stellar_flux (luminosity, distance):
    """Returns stellar flux"""
    return luminosity / (4 * PI * distance**2)

The 0D balance condition shall be encoded now
$$
C \frac{dT}{dt}= \frac{1}{4} (1-\alpha)S_0 - \epsilon \sigma T^4
$$

At radiative equilibrium; $\frac{dT}{dt} = 0$ and thus we get
$$
\epsilon \sigma T^4 = \frac{1}{4} (1-\alpha)S_0
$$

In [7]:
def equilibrium_temperature (stellar_flux, albedo, emissivity):
    """Returns the equilibrium temperature of a planet"""
    return (stellar_flux * (1 - albedo) / (4 * sigma * emissivity))**(1/4)

In [8]:
S0 = stellar_flux(L_star, Orbital_radius)
T_eq = equilibrium_temperature(S0, alpha, epsilon0)

print(f"S0 = {S0:.2f} W/m²")
print(f"T_eq = {T_eq:.2f} K")

S0 = 1361.13 W/m²
T_eq = 271.77 K


29/06/2026
Adding Greenhouse Effect

Obsv:
343.99K at 0.3 emissivity, that's too hot.

Instead of just tuning the parameter, we shall introduce layers of atmosphere. The shell models.

Obsv:
$T_{s}$ comes out to be 287.48K for 30% Albedo and 77% emissivity, which is accurate for Earth.

-------
Since previous functions cannot incorporate a shell, we introduce a new function following,
$$
\frac{(1-\alpha)S_0}{4} + \epsilon \sigma T_{a}^4 = \sigma T_{s}^4
$$

Here, $T_a$ and $T_s$ are the temperatures of the atmosphere and the surface respectively. This is the **Surface Energy Balance** Equation.

Due to Greenhouse gases affecting the emissivity, we must also do an energy balance on the atmosphere.
$$
\epsilon \sigma T_{s}^4 = 2\epsilon \sigma T_{a}^4
$$

This is the **Atmospheric Energy Balance** Equation.

We are only interested in the $T_{s}^4$, surface temperature, which after solving the two equations is given by the following
$$
T_{s}^4 = \left(\frac{(1-\alpha)S_{0}}{4 \sigma (1-\frac{\epsilon}{2})} \right)^\frac{1}{4}
$$

In [9]:
def surface_temp_w_greenhouse (S0, alpha, epsilon0):
    return (
        ((1-alpha)*S0)
        / (4 * sigma * (1-epsilon0/2))
    )**0.25

print(f"T_s = {surface_temp_w_greenhouse(S0, alpha, epsilon0):.2f} K")

T_s = 287.48 K


### Step 2: Adding an Atmosphere

09/07/2026
We make emissivity a function of composition and pressure to make amore realistic atmosphere.

Unfortunately, to model all the effects of the atmospheric composition on the climate, the chemistry gets sophisticated, therefore our model is limited to the effects of composition on the Greenhouse Effect only.

We use optical depth $(\tau)$ to model the emissivity $(\epsilon)$. Initially $\tau$ is treated as a constant, and then we replace it with a physically motivated expression based on composition and pressure of the atmospheric gases.

In [10]:
tau_old = 0.8  # Optical depth of the atmosphere
epsilon_old = 1 - np.exp(-tau_old)  # Emissivity of the planet

print(f"Emissivity (epsilon) = {epsilon_old:.2f}")
print(f"T_s with gray gas approximation = {surface_temp_w_greenhouse(S0, alpha, epsilon_old):.2f} K")

Emissivity (epsilon) = 0.55
T_s with gray gas approximation = 275.93 K


#### Step 2.1: Optical Depth Function

04/08/2026
Now we use the compostion and partial pressure of the gases and the gravity of the planet, to make a physical equation for optical depth.

The physics invloved is as follows:

For each gas,
$$
P_i = x_i P
$$

and,
$$
m_i = \frac{P_i}{g}
$$
where $m_i$ is the atmospheric mass of a gas above one square meter.

then, the optical density
$$
\tau_{i} = \kappa_{i} m_i
$$

$$
\tau = \sum_{i} \tau_{i}
$$

In [ ]:
composition = {
    "N2": 0.7808,
    "O2": 0.2095,
    "Ar": 0.0093,
    "CO2": 0.0004,
}

gravity = 9.81  # m/s²

pressure = 101325  # Pa

kappa = {
    "N2": 0.0001,
    "O2": 0.0001,
    "Ar": 0.0001,
    "CO2": 0.0004,
} # values are unconfirmed and for demonstration purposes only

In [12]:
def optical_depth(pressure, gravity, composition, kappa):
    """Returns the optical depth of the atmosphere"""
    tau = 0.0
    tau_species = {}

    for gas, x in composition.items():

        P_i = x * pressure          # Partial pressure
        m_i = P_i / gravity         # Column mass
        tau_i = kappa[gas] * m_i    # Optical depth

        tau_species[gas] = tau_i
        tau += tau_i

        return tau, tau_species

epsilon = 1 - np.exp(-optical_depth(pressure, gravity, composition, kappa)[0])

print(f"Optical depth (tau) = {optical_depth(pressure, gravity, composition, kappa)[0]:.2f}")
print(f"Emissivity (epsilon) = {epsilon:.2f}")
print(f"T_s with physical tau = {surface_temp_w_greenhouse(S0, alpha, epsilon):.2f} K")

Optical depth (tau) = 0.81
Emissivity (epsilon) = 0.55
T_s with physical tau = 276.07 K
